# Pandas: GroupBy, Aggregation, Merge, Concat, and Reshaping 

The operations used to summarize datasets, combine multiple tables, and reshape data structures. These skills are critical for: 
- Exploratory data analysis 
- Data cleaning 
- Preparing datasets for machine learning workflows 
- Working with relational-style data

In [27]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 20)

df = pd.read_csv("data/users.csv")
df.head()


,id,name,age,city,score,signup_date,is_active
0,1,Alice,25.0,New York,88.0,2023-01-05,True
1,2,Bob,30.0,Los Angeles,92.0,2023-01-10,False
2,3,Charlie,NaN,Chicago,75.0,2023-01-12,True
3,4,David,28.0,NaN,81.0,2023-01-15,True
4,5,Eva,22.0,Houston,NaN,2023-01-20,False


## GroupBy Basics 

The `groupby()` operation splits the data into groups based on one or more columns. Common syntax: 

In [28]:
df.groupby("city")["age"].mean()

city
Atlanta          29.750000
Austin           29.000000
Boston           27.500000
Charlotte        27.250000
Chicago          30.666667
                   ...    
San Antonio      31.250000
San Diego        26.750000
San Francisco    26.250000
Seattle          26.600000
Washington DC    27.000000
Name: age, Length: 24, dtype: float64

Above example should be understood like this: 
- First, we group the data within column "city"
- Then, calculate the mean of age throughout this grouped data

### GroupBy count

In [29]:
df.groupby("city")["city"].count()

city
Atlanta          6
Austin           6
Boston           6
Charlotte        6
Chicago          7
                ..
San Antonio      6
San Diego        6
San Francisco    6
Seattle          6
Washington DC    6
Name: city, Length: 24, dtype: int64

### Multiple Aggregations with agg()

We can compute multiple statistics for one or many columns simultaneously

In [30]:
df.groupby("city").agg({
    "age": ["mean", "max", "min"]
})

age            
                    mean   max   min
city                                
Atlanta        29.750000  35.0  25.0
Austin         29.000000  33.0  27.0
Boston         27.500000  30.0  24.0
Charlotte      27.250000  29.0  23.0
Chicago        30.666667  34.0  27.0
...                  ...   ...   ...
San Antonio    31.250000  33.0  29.0
San Diego      26.750000  30.0  25.0
San Francisco  26.250000  28.0  23.0
Seattle        26.600000  30.0  21.0
Washington DC  27.000000  30.0  24.0

[24 rows x 3 columns]

### `transform` vs `agg`
- `agg()` returns a **reduced** DataFrame (fewer rows)
- `transform()` returns a Series with **same number of rows as the original**

In [31]:
df["mean_age_by_city"] = df.groupby("city")["age"].transform("mean")
df[["city", "age", "mean_age_by_city"]].head()

,city,age,mean_age_by_city
0,New York,25.0,26.800000
1,Los Angeles,30.0,27.000000
2,Chicago,NaN,30.666667
3,NaN,28.0,NaN
4,Houston,22.0,24.750000


## Merge (Joins)

`pd.merge()` combines DataFrames on one or more keys

Types of joins: 
- `inner` -> keep matching rows only 
- `left` -> keep all rows from left table 
- `right`
- `outer` -> keep all rows from both tables

Syntax: 

```
pd.merge(df1, df2, on="key", how="inner")
```

In [32]:
left = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["John", "Anna", "Peter"]
})

right = pd.DataFrame({
    "id": [2, 3, 4],
    "salary": [50000, 60000, 70000]
})

left, right


(   id   name
 0   1   John
 1   2   Anna
 2   3  Peter,
    id  salary
 0   2   50000
 1   3   60000
 2   4   70000)

In [33]:
pd.merge(left,right ,on="id", how="inner")

,id,name,salary
0,2,Anna,50000
1,3,Peter,60000


`how="inner"` = Inner Join -> keep only ids that appear in BOTH tables

In [34]:
pd.merge(left, right, on="id", how="left")

,id,name,salary
0,1,John,NaN
1,2,Anna,50000.0
2,3,Peter,60000.0


`how="left"` = Left join -> keep all rows from LEFT table

In [35]:
pd.merge(left, right, on="id", how="outer")

,id,name,salary
0,1,John,NaN
1,2,Anna,50000.0
2,3,Peter,60000.0
3,4,NaN,70000.0


`how="outer"` = Outer join -> Keep all rows from BOTH tables

## Concat (Stacking DataFrames)

`pd.concat()` stacks DataFrames: 
- **Vertically** (`axis=0`): add more rows
- **Horizontally** (`axis=1`): add more columns

In [36]:
df1 = df.iloc[:3]
df2 = df.iloc[3:6]

pd.concat([df1, df2], axis=0)


,id,name,age,city,score,signup_date,is_active,mean_age_by_city
0,1,Alice,25.0,New York,88.0,2023-01-05,True,26.800000
1,2,Bob,30.0,Los Angeles,92.0,2023-01-10,False,27.000000
2,3,Charlie,NaN,Chicago,75.0,2023-01-12,True,30.666667
3,4,David,28.0,NaN,81.0,2023-01-15,True,NaN
4,5,Eva,22.0,Houston,NaN,2023-01-20,False,24.750000
5,6,Frank,27.0,Phoenix,79.0,NaN,True,27.750000


In [37]:
part1 = df[["age", "city"]].head()
part2 = df[["mean_age_by_city"]].head()

pd.concat([part1, part2], axis=1)


,age,city,mean_age_by_city
0,25.0,New York,26.800000
1,30.0,Los Angeles,27.000000
2,NaN,Chicago,30.666667
3,28.0,NaN,NaN
4,22.0,Houston,24.750000


## Reshaping (Pivot/Melt)

### Pivot Tables 

A pivot table reshapes data by turning unique values of a column into new columns 

Syntax: 

```
df.pivot_table(index="row_key", columns="column_key", values="value_col", aggfunc="mean")
```

Explaination: 
- `index` -> Choosing what to group by 
- `columns` -> Choosing what becomes columns 
- `values` -> Selecting the values to summarizes 
- `aggfunc` -> Deciding how to summarize them 

In [38]:
df.pivot_table(
    index="city",
    values="score",
    aggfunc="mean"
)


,score
city,
Atlanta,83.000000
Austin,87.666667
Boston,88.666667
Charlotte,78.600000
Chicago,84.000000
...,...
San Antonio,84.750000
San Diego,80.500000
San Francisco,74.333333


In [39]:
df.pivot_table(
    index="city",
    columns="is_active", # True or False 
    values="id", # Apply agg function to this columns 
    aggfunc="count",
    fill_value=0
)


is_active,False,True
city,,
Atlanta,5,1
Austin,5,1
Boston,0,6
Charlotte,1,5
Chicago,1,6
...,...,...
San Antonio,5,1
San Diego,0,6
San Francisco,1,5


### Melt - Unpivoting data (Wide -> Long)

Turns columns into rows

Syntax: 

```
pd.melt(df, id_vars="id_column")
```

In [40]:
wide_df = pd.DataFrame({
    "id": [1, 2],
    "math": [80, 90],
    "english": [85, 88]
})

wide_df


,id,math,english
0,1,80,85
1,2,90,88


In [41]:
pd.melt(wide_df, id_vars="id", var_name="subject", value_name="score")


,id,subject,score
0,1,math,80
1,2,math,90
2,1,english,85
3,2,english,88
